In [ ]:
!pip install optuna

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SqueezeExciteBlock(nn.Module):
    def __init__(self, channels, ratio=16):
        super(SqueezeExciteBlock, self).__init__()
        self.fc1 = nn.Conv1d(channels, channels // ratio, kernel_size=1, bias=False)
        self.fc2 = nn.Conv1d(channels // ratio, channels, kernel_size=1, bias=False)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        se = F.adaptive_avg_pool1d(x, 1)  # (B, C, 1)
        se = self.fc1(se)
        se = self.relu(se)
        se = self.fc2(se)
        se = self.sigmoid(se)
        return x * se  # Broadcast a lo largo de T

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

class ConvSEBlock(nn.Module):
    """
    Bloque genérico de Conv1d + BatchNorm + ReLU + (opcional) Squeeze-and-Excitation.
    """
    def __init__(self,
                 in_channels: int,
                 out_channels: int,
                 kernel_size: int,
                 se_ratio: int = 16,
                 use_se: bool = True,
                 padding: Optional[str] = 'same',
                 bias: bool = False):
        super().__init__()
        self.conv = nn.Conv1d(
            in_channels, out_channels,
            kernel_size=kernel_size,
            padding=padding,
            bias=bias
        )
        self.bn = nn.BatchNorm1d(out_channels)
        self.use_se = use_se
        if use_se:
            self.se = SqueezeExciteBlock(out_channels, ratio=se_ratio)
        else:
            self.se = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        x = self.bn(x)
        x = F.relu(x)
        if self.use_se and self.se is not None:
            x = self.se(x)
        return x

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List

########################################
# Clase Base: DisjointBase
########################################
class DisjointBase(nn.Module):
    def __init__(self, in_height: int, out_filters: int, temporal_kernel: int, activation: nn.Module = nn.ELU):
        """
        Bloque básico que opera en dos etapas:
          1. Convolución temporal (con padding asimétrico) para extraer características locales en el tiempo.
          2. Convolución espacial para colapsar la dimensión de variates.

        Parámetros:
          - in_height: número de variates (o “altura”) de la entrada.
          - out_filters: número de filtros de salida del bloque temporal.
          - temporal_kernel: tamaño del kernel que opera en la dimensión temporal.
          - activation: función de activación (por defecto nn.ELU).
        """
        super().__init__()
        self.temporal_kernel = temporal_kernel

        # Calcular padding asimétrico para preservar la dimensión temporal T.
        if temporal_kernel % 2 == 0:
            self.pad_left = temporal_kernel // 2 - 1
            self.pad_right = temporal_kernel // 2
        else:
            self.pad_left = self.pad_right = temporal_kernel // 2

        # Bloque temporal: la convolución opera sobre la dimensión temporal.
        self.temp_conv = nn.Conv2d(
            in_channels=1,
            out_channels=out_filters,
            kernel_size=(1, temporal_kernel),
            padding=(0, 0)
        )
        self.temp_bn = nn.BatchNorm2d(out_filters)
        self.temp_act = activation(alpha=1.0) if hasattr(activation, '__call__') else activation()

        # Bloque espacial: se colapsa la dimensión de variates utilizando un kernel que cubre toda la altura.
        self.spatial_conv = nn.Conv2d(
            in_channels=out_filters,
            out_channels=out_filters,
            kernel_size=(in_height, 1),
            padding=0
        )
        self.spatial_bn = nn.BatchNorm2d(out_filters)
        self.spatial_act = activation(alpha=1.0) if hasattr(activation, '__call__') else activation()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parámetros:
          - x: tensor de forma (B, in_height, T)
        Retorna:
          - Tensor de forma (B, out_filters, T)
        """
        # Expandir dimensión para poder aplicar Conv2d: (B, 1, in_height, T)
        x = x.unsqueeze(1)
        # Aplicar padding asimétrico en la dimensión temporal
        x = F.pad(x, (self.pad_left, self.pad_right, 0, 0))

        # Bloque temporal
        x = self.temp_conv(x)         # (B, out_filters, in_height, T)
        x = self.temp_bn(x)
        x = self.temp_act(x)

        # Bloque espacial (colapsa la dimensión "in_height")
        x = self.spatial_conv(x)      # (B, out_filters, 1, T)
        x = self.spatial_bn(x)
        x = self.spatial_act(x)

        # Eliminar la dimensión de tamaño 1 (la "altura")
        x = x.squeeze(2)              # (B, out_filters, T)
        return x

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Union

class ResBlock(nn.Module):
    """
    Bloque residual simple para redes 1D.

    Este bloque consta de tres capas convolucionales 1D, cada una seguida de Batch Normalization y activación ReLU.
    Además, se incorpora una conexión de atajo (shortcut) para facilitar el flujo del gradiente, la cual se ajusta mediante
    una convolución 1x1 si la dimensión de entrada no coincide con la de salida o si se aplica un stride diferente de 1.

    Parámetros:
      - in_channels (int): Número de canales de entrada.
      - out_channels (int): Número de canales de salida.
      - stride (int, opcional): Stride a utilizar en la primera convolución (por defecto 1).
    """
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1) -> None:
        super().__init__()
        # Primera convolución: kernel de tamaño 5
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=5, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm1d(out_channels)

        # Segunda convolución: kernel de tamaño 3
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm1d(out_channels)

        # Tercera convolución: kernel de tamaño 3
        self.conv3 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn3   = nn.BatchNorm1d(out_channels)

        self.relu = nn.ReLU(inplace=True)

        # Shortcut: se ajusta la dimensión de entrada si es necesario
        if in_channels != out_channels or stride != 1:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out += identity
        return self.relu(out)


class ConvBNReLU(nn.Module):
    """
    Sub-bloque que aplica Batch Normalization, seguida de una activación ReLU y una convolución 1D,
    utilizando padding para preservar la longitud temporal.

    Parámetros:
      - in_channels (int): Número de canales de entrada.
      - out_channels (int): Número de canales de salida.
      - kernel_size (int, opcional): Tamaño del kernel convolucional (por defecto 3).
    """
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3) -> None:
        super().__init__()
        self.bn    = nn.BatchNorm1d(in_channels)
        self.relu  = nn.ReLU(inplace=True)
        self.conv  = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, padding=kernel_size // 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.bn(x)
        x = self.relu(x)
        x = self.conv(x)
        return x


class ResBlock3x(nn.Module):
    """
    Bloque residual compuesto por tres sub-bloques consecutivos de la forma:
      BatchNorm → ReLU → Convolución 1D.

    Al final se suma la entrada (o se ajusta mediante una proyección 1x1 si es necesario) para crear la conexión residual.

    Parámetros:
      - in_channels (int): Número de canales de entrada.
      - out_channels (int): Número de canales de salida.
    """
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.subblock1 = ConvBNReLU(in_channels, out_channels)
        self.subblock2 = ConvBNReLU(out_channels, out_channels)
        self.subblock3 = ConvBNReLU(out_channels, out_channels)

        # Ajuste de la dimensión de la entrada si es necesario
        self.skip_conv = nn.Conv1d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.skip_conv(x)
        out = self.subblock1(x)
        out = self.subblock2(out)
        out = self.subblock3(out)
        out += identity
        return out

# PeepholeLSTM

In [ ]:
import math
import torch
import torch.nn as nn
from typing import Optional

# ──────────────────────────────────────────────────────────────────────────────
#  Peephole LSTM Cell (Implementación Final Corregida)
# ──────────────────────────────────────────────────────────────────────────────
class PeepholeLSTMCell(nn.Module):
    """Célula LSTM con conexiones peephole."""
    def __init__(self, input_size: int, hidden_size: int, bias: bool = True):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.bias = bias  # <--- Variable de instancia crítica

        # Parámetros
        self.W_x = nn.Parameter(torch.Tensor(4 * hidden_size, input_size))
        self.W_h = nn.Parameter(torch.Tensor(4 * hidden_size, hidden_size))
        self.W_ci = nn.Parameter(torch.Tensor(hidden_size))
        self.W_cf = nn.Parameter(torch.Tensor(hidden_size))
        self.W_co = nn.Parameter(torch.Tensor(hidden_size))

        if self.bias:  # <--- Usar self.bias en lugar de bias
            self.b = nn.Parameter(torch.Tensor(4 * hidden_size))
        else:
            self.register_parameter('b', None)

        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1.0 / math.sqrt(self.hidden_size)
        for p in self.parameters():
            nn.init.uniform_(p, -stdv, stdv)

    def forward(self, x, hx):
        h_prev, c_prev = hx
        gates = (x @ self.W_x.t()) + (h_prev @ self.W_h.t())
        if self.bias:  # <--- Acceso correcto al atributo
            gates += self.b

        i, f, g, o = gates.chunk(4, 1)
        i = torch.sigmoid(i + c_prev * self.W_ci)
        f = torch.sigmoid(f + c_prev * self.W_cf)
        g = torch.tanh(g)
        c = f * c_prev + i * g
        o = torch.sigmoid(o + c * self.W_co)
        h = o * torch.tanh(c)
        return h, c


class PeepholeLSTM(nn.Module):
    """LSTM bidireccional con peephole connections implementado correctamente."""
    def __init__(self,
                 time_steps: int,
                 input_channels: int,
                 output_channels: int,
                 hidden_size: int = 64,
                 num_layers: int = 1,
                 bidirectional: bool = False):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        # Crear células por dirección
        self.cells = nn.ModuleList()
        for _ in range(self.num_directions):
            layer_cells = nn.ModuleList()
            in_size = input_channels
            for _ in range(num_layers):
                layer_cells.append(PeepholeLSTMCell(in_size, hidden_size))
                in_size = hidden_size * self.num_directions  # Ajuste para capas profundas
            self.cells.append(layer_cells)

        # Capa de proyección final
        self.fc = nn.Conv1d(
            in_channels=hidden_size * self.num_directions,
            out_channels=output_channels,
            kernel_size=1
        )

    def forward(self, x):
        # x: (B, C, T) → (T, B, C)
        x = x.permute(2, 0, 1)
        seq_len, batch_size, _ = x.shape

        outputs = []
        for direction in range(self.num_directions):
            x_dir = x.flip(0) if direction == 1 else x
            h = [x.new_zeros(batch_size, self.hidden_size) for _ in range(self.num_layers)]
            c = [x.new_zeros(batch_size, self.hidden_size) for _ in range(self.num_layers)]

            direction_outputs = []
            for t in range(seq_len):
                inp = x_dir[t]
                for layer in range(self.num_layers):
                    # Corrección clave: paréntesis cerrado
                    h_layer, c_layer = self.cells[direction][layer](inp, (h[layer], c[layer]))
                    h[layer] = h_layer
                    c[layer] = c_layer
                    inp = h_layer
                    # Concatenar solo si hay más de una capa y es bidireccional
                    if layer < self.num_layers - 1 and self.bidirectional:
                        inp = torch.cat([inp, h[layer]], dim=1)
                direction_outputs.append(h_layer)

            if direction == 1:
                direction_outputs = direction_outputs[::-1]
            outputs.append(torch.stack(direction_outputs, dim=2))

        y = torch.cat(outputs, dim=1) if self.bidirectional else outputs[0]
        return self.fc(y)


# PhasedLSTM

In [ ]:
import math
import torch
import torch.nn as nn
from typing import Optional

# ──────────────────────────────────────────────────────────────────────────────
#  Phased LSTM Cell + Wrapper
# ──────────────────────────────────────────────────────────────────────────────
class PhasedLSTMCell(nn.Module):
    """
    Phased LSTM Cell (Neil et al., 2016):
      - incorpora un 'time gate' k_t definido por periodo, fase y ratio_on.
      - requiere timestamps en entrada.
    """
    def __init__(self,
                 input_size: int,
                 hidden_size: int,
                 period_init: float = 1.0,
                 r_on_init: float = 0.05,
                 bias: bool = True):
        super().__init__()
        self.input_size  = input_size
        self.hidden_size = hidden_size

        # pesos estándar LSTM (sin peephole)
        self.W_x = nn.Parameter(torch.Tensor(4 * hidden_size, input_size))
        self.W_h = nn.Parameter(torch.Tensor(4 * hidden_size, hidden_size))
        self.b   = nn.Parameter(torch.Tensor(4 * hidden_size)) if bias else None

        # parámetros del time gate
        self.period = nn.Parameter(torch.Tensor(hidden_size))
        self.phase  = nn.Parameter(torch.Tensor(hidden_size))
        self.r_on   = nn.Parameter(torch.Tensor(hidden_size))

        self.reset_parameters(period_init, r_on_init)

    def reset_parameters(self, period_init, r_on_init):
        stdv = 1.0 / math.sqrt(self.hidden_size)
        for p in [self.W_x, self.W_h]:
            nn.init.uniform_(p, -stdv, stdv)
        if self.b is not None:
            nn.init.uniform_(self.b, -stdv, stdv)
        # inicializar periodos y fases aleatoriamente
        nn.init.uniform_(self.period, 0.5 * period_init, 1.5 * period_init)
        nn.init.uniform_(self.phase, 0.0, period_init)
        nn.init.constant_(self.r_on, r_on_init)

    def forward(self, x, hx, timestamp):
        """
        x:          (B, input_size)
        hx:         (h_prev, c_prev), cada uno (B, hidden_size)
        timestamp:  (B,) tiempo continuo para cada muestra
        """
        h_prev, c_prev = hx
        # standard LSTM gates
        gates = x @ self.W_x.t() + h_prev @ self.W_h.t()
        if self.b is not None:
            gates = gates + self.b
        i, f, g, o = gates.chunk(4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        g = torch.tanh(g)
        c = f * c_prev + i * g
        o = torch.sigmoid(o)

        # compute phase of each neuron
        # ϕ = ((t - phase) mod period) / period
        phi = ((timestamp.unsqueeze(1) - self.phase) % self.period) / self.period
        # time gate k_t
        k = torch.where(
            phi < self.r_on / 2,
            2 * phi / self.r_on,
            torch.where(
                phi < self.r_on,
                2 - 2 * phi / self.r_on,
                torch.zeros_like(phi)
            )
        )
        # output update only when gate open
        h = k * (o * torch.tanh(c)) + (1 - k) * h_prev
        return h, c


class PhasedLSTM(nn.Module):
    def __init__(self,
                 time_steps: int,
                 input_channels: int,
                 output_channels: int,
                 hidden_size: int = 64,
                 num_layers: int = 1,
                 period_init: float = 1.0,    # Nuevo parámetro
                 r_on_init: float = 0.05):    # Nuevo parámetro
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.cells = nn.ModuleList([
            PhasedLSTMCell(
                input_channels if l == 0 else hidden_size,
                hidden_size,
                period_init=period_init,  # Pasa los hiperparámetros a la célula
                r_on_init=r_on_init
            )
            for l in range(num_layers)
        ])
        self.fc = nn.Conv1d(hidden_size, output_channels, kernel_size=1)

    def forward(self, x, times):
        # x:     (B, C, T)
        # times: (B, T) timestamps in same units que period
        x = x.permute(2, 0, 1)      # → (T, B, C)
        times = times.permute(1, 0) # → (T, B)
        T, B, _ = x.size()

        h = [x.new_zeros(B, self.hidden_size) for _ in range(self.num_layers)]
        c = [x.new_zeros(B, self.hidden_size) for _ in range(self.num_layers)]
        outputs = []

        for t in range(T):
            inp = x[t]
            ts  = times[t]
            for l, cell in enumerate(self.cells):
                h_l, c_l = cell(inp, (h[l], c[l]), ts)
                h[l], c[l] = h_l, c_l
                inp = h_l
            outputs.append(h_l)

        y = torch.stack(outputs, dim=2)  # (B, hidden_size, T)
        return self.fc(y)

# ESN

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 3) Echo State Network (ESN)
# ──────────────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F

class ESN(nn.Module):
    """
    Echo State Network:
      - W_in, W_res fijos y aleatorios
      - sólo W_out se entrena
      - ridge_alpha para regularización Ridge en W_out (por implementar en entrenamiento)
    """
    def __init__(self,
                 time_steps, input_channels, output_channels,
                 reservoir_size: int = 500,
                 spectral_radius: float = 0.9,
                 sparsity: float = 0.1,
                 bias: bool = True,
                 ridge_alpha: float = 1e-3):
        super().__init__()
        self.res_size = reservoir_size
        self.ridge_alpha = ridge_alpha  # ahora aceptado

        # Entrada → reserva
        self.W_in  = nn.Parameter(
            torch.empty(reservoir_size, input_channels).normal_(0, 1),
            requires_grad=False
        )

        # Matriz recurrente dispersa
        W = torch.randn(reservoir_size, reservoir_size)
        mask = torch.rand(reservoir_size, reservoir_size) < sparsity
        W *= mask.float()
        # ajustar radio espectral
        eigs = torch.linalg.eigvals(W)
        radius = eigs.abs().max()
        W *= (spectral_radius / radius)
        self.W_res = nn.Parameter(W, requires_grad=False)

        # lectura (entrenable)
        self.W_out = nn.Parameter(
            torch.empty(output_channels, reservoir_size).uniform_(-0.1, 0.1)
        )
        self.b     = nn.Parameter(torch.zeros(output_channels)) if bias else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, C_in, T)
        """
        B, _, T = x.size()
        h = x.new_zeros(B, self.res_size)
        states = []
        for t in range(T):
            u = x[:, :, t]  # (B, C_in)
            h = torch.tanh(
                F.linear(u, self.W_in) +
                h @ self.W_res.t()
            )
            states.append(h)
        S = torch.stack(states, dim=2)           # (B, res_size, T)

        # proyectar con W_out
        # F.linear quiere (B*T, res_size), pero con permisos funciona:
        Y = F.linear(S.permute(0, 2, 1), self.W_out, self.b)  # (B, T, C_out)
        return Y.permute(0, 2, 1)                   # (B, C_out, T)


# IndRNN

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ──────────────────────────────────────────────────────────────────────────────
# 1) Independently RNN (IndRNN)
# ──────────────────────────────────────────────────────────────────────────────
class IndRNNLayer(nn.Module):
    """
    Capa IndRNN: cada neurona tiene W_rec escalar propio.
    """
    def __init__(self, input_size: int, hidden_size: int, bias: bool = True):
        super().__init__()
        self.hidden_size = hidden_size
        self.W_in  = nn.Parameter(torch.Tensor(hidden_size, input_size))
        self.W_rec = nn.Parameter(torch.Tensor(hidden_size))
        if bias:
            self.b = nn.Parameter(torch.Tensor(hidden_size))
        else:
            self.register_parameter('b', None)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.W_in, a=math.sqrt(5))
        # recurrent weights positivos para estabilidad
        nn.init.uniform_(self.W_rec, 0, 1)
        if self.b is not None:
            fan_in = self.W_in.size(1)
            bound = 1 / math.sqrt(fan_in)
            nn.init.uniform_(self.b, -bound, bound)

    def forward(self, x, hx=None, clip_max: float = None):
        """
        x: (B, T, C_in)
        hx: (B, hidden_size)
        clip_max: si no es None, clipea W_rec en [-clip_max, clip_max]
        """
        if clip_max is not None:
            with torch.no_grad():
                self.W_rec.data.clamp_(-clip_max, clip_max)

        B, T, _ = x.size()
        if hx is None:
            hx = x.new_zeros(B, self.hidden_size)
        outputs = []
        h = hx
        for t in range(T):
            xt = x[:, t, :]                        # (B, C_in)
            h = F.linear(xt, self.W_in) + h * self.W_rec
            if self.b is not None:
                h = h + self.b
            h = torch.relu(h)
            outputs.append(h)
        y = torch.stack(outputs, dim=1)            # (B, T, hidden_size)
        return y, h


class IndRNN(nn.Module):
    """
    Red IndRNN multicapa con dropout intermedio y clipping de W_rec.
    """
    def __init__(self,
                 time_steps: int,
                 input_channels: int,
                 output_channels: int,
                 hidden_size: int = 64,
                 num_layers: int = 1,
                 dropout_rate: float = 0.0,
                 clip_max: float = 1.0):
        super().__init__()
        self.num_layers = num_layers
        self.clip_max   = clip_max
        self.time_steps = time_steps

        # Construir capas IndRNN
        layers = []
        in_size = input_channels
        for _ in range(num_layers):
            layers.append(IndRNNLayer(in_size, hidden_size))
            in_size = hidden_size
        self.rnn_layers = nn.ModuleList(layers)
        self.dropout    = nn.Dropout(dropout_rate)

        # Capa de salida 1×1 conv para mantener formato (B, C_out, T)
        self.fc = nn.Conv1d(hidden_size, output_channels, kernel_size=1)

    def forward(self, x):
        # x: (B, C_in, T) → permute a (B, T, C_in)
        y = x.permute(0, 2, 1)
        h = None
        # Propagar por cada capa
        for layer in self.rnn_layers:
            y, h = layer(y, hx=h, clip_max=self.clip_max)
            y = self.dropout(y)  # dropout entre capas

        # y: (B, T, hidden_size) → (B, hidden_size, T)
        y = y.permute(0, 2, 1)
        return self.fc(y)


# QRNN

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ──────────────────────────────────────────────────────────────────────────────
#  QRNN layer con opcional puerta de salida
# ──────────────────────────────────────────────────────────────────────────────
class QRNNLayer(nn.Module):
    def __init__(self,
                 input_channels: int,
                 hidden_size: int,
                 kernel_size: int = 2,
                 pool_type: str = "f",  # "f", "o" o "fo"
                 bias: bool = True):
        super().__init__()
        padding = kernel_size - 1
        self.pool_type = pool_type
        # Z = actualización
        self.conv_z = nn.Conv1d(input_channels, hidden_size,
                                kernel_size, padding=padding, bias=bias)
        # F = olvido
        self.conv_f = nn.Conv1d(input_channels, hidden_size,
                                kernel_size, padding=padding, bias=bias)
        # O = salida, sólo si me pidieron "o" u "fo"
        if pool_type in ("o", "fo"):
            self.conv_o = nn.Conv1d(input_channels, hidden_size,
                                    kernel_size, padding=padding, bias=bias)
        else:
            self.conv_o = None

    def forward(self, x: torch.Tensor):
        """
        x: (B, C, T)
        returns y: (B, H, T)
        """
        T = x.size(2)
        Z = torch.tanh(self.conv_z(x))[:, :, :T]
        Fg = torch.sigmoid(self.conv_f(x))[:, :, :T]
        if self.conv_o is not None:
            Og = torch.sigmoid(self.conv_o(x))[:, :, :T]
        else:
            Og = None

        B, H, _ = Z.size()
        h = x.new_zeros(B, H)
        y = []
        for t in range(T):
            zt = Z[:, :, t]
            ft = Fg[:, :, t]
            h = ft * h + (1 - ft) * zt
            if Og is not None:
                # h_tilde = h, luego salida = o_t * h
                h = Og[:, :, t] * h
            y.append(h)
        y = torch.stack(y, dim=2)  # (B, H, T)
        return y


# ──────────────────────────────────────────────────────────────────────────────
#  QRNN multicapa + dropout + 1×1 conv de salida
# ──────────────────────────────────────────────────────────────────────────────
class QRNN(nn.Module):
    def __init__(self,
                 time_steps: int,
                 input_channels: int,
                 output_channels: int,
                 hidden_size: int = 64,
                 kernel_size: int = 2,
                 num_layers: int = 1,
                 dropout_rate: float = 0.0,
                 pool_type: str = "f"):
        super().__init__()
        self.time_steps = time_steps

        layers = []
        in_ch = input_channels
        for _ in range(num_layers):
            layers.append(QRNNLayer(in_ch, hidden_size,
                                    kernel_size=kernel_size,
                                    pool_type=pool_type))
            in_ch = hidden_size
        self.rnn_layers = nn.ModuleList(layers)
        self.dropout    = nn.Dropout(dropout_rate)
        self.fc         = nn.Conv1d(hidden_size, output_channels, kernel_size=1)

    def forward(self, x: torch.Tensor):
        # x: (B, C_in, T)
        y = x
        for layer in self.rnn_layers:
            y = layer(y)            # (B, hidden, T)
            y = self.dropout(y)
        return self.fc(y)           # (B, C_out, T)


In [ ]:
import torch
import torch.nn as nn
from transformers import BertModel

class BERT_BiGRU(nn.Module):
    def __init__(self, bert_model_name='bert-base-uncased', hidden_dim=100,
                 num_classes=2, dropout_prob=0.2, num_layers=2):
        """
        BERT-BiGRU model for text classification.
        """
        super(BERT_BiGRU, self).__init__()

        self.bert = BertModel.from_pretrained(bert_model_name)
        bert_hidden_dim = self.bert.config.hidden_size

        self.bigru = nn.GRU(
            input_size=bert_hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout_prob if num_layers > 1 else 0,
            bidirectional=True,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids=None, attention_mask=None, batch=None):
        """
        Forward pass. Accepts either:
        - input_ids and attention_mask explicitly, or
        - a batch dictionary containing both.
        """
        if batch is not None:
            input_ids = batch['input_ids']
            attention_mask = batch['attention_mask']

        if input_ids is None or attention_mask is None:
            raise ValueError("Either (input_ids and attention_mask) or batch must be provided.")

        # BERT encoding
        bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = bert_outputs.last_hidden_state  # (B, L, H)

        # BiGRU
        gru_output, _ = self.bigru(sequence_output)
        last_hidden = gru_output[:, -1, :]  # (B, 2*H)

        # Dropout and classification
        dropped = self.dropout(last_hidden)
        logits = self.classifier(dropped)
        return logits


In [ ]:
import torch
import torch.nn as nn
import math
from typing import Optional

class PositionalEncoding(nn.Module):
    """Codificación posicional senoidal clásica (Vaswani et al., 2017)."""
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)  # (T, D)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # (T, 1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))  # (D//2,)
        pe[:, 0::2] = torch.sin(position * div_term)  # pares
        pe[:, 1::2] = torch.cos(position * div_term)  # impares
        pe = pe.unsqueeze(0).transpose(1, 2)  # (1, D, T)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, D, T)
        T = x.size(-1)
        return x + self.pe[:, :, :T]


class Transformer1D(nn.Module):
    """
    Transformer adaptado a series temporales multivariadas:
    Entrada:  (B, C_in, T)
    Salida:   (B, C_out, T)
    """
    def __init__(
        self,
        time_steps: int,
        input_channels: int,
        output_channels: int,
        d_model: int = 128,
        num_heads: int = 4,
        num_layers: int = 3,
        dropout: float = 0.1,
        **kwargs  # para compatibilidad con ModelFactory
    ):
        super().__init__()
        self.input_proj = nn.Conv1d(input_channels, d_model, kernel_size=1)
        self.pos_enc = PositionalEncoding(d_model, max_len=time_steps)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.output_proj = nn.Conv1d(d_model, output_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C_in, T)
        x = self.input_proj(x)            # (B, D, T)
        x = self.pos_enc(x)               # (B, D, T)
        x = x.permute(0, 2, 1)            # (B, T, D)
        x = self.encoder(x)               # (B, T, D)
        x = x.permute(0, 2, 1)            # (B, D, T)
        return self.output_proj(x)        # (B, C_out, T)


In [13]:
import torch
import torch.nn as nn
import math

class PatchEmbedding(nn.Module):
    def __init__(self, input_channels, patch_len, stride, d_model):
        super().__init__()
        self.patch_embed = nn.Conv1d(input_channels, d_model, kernel_size=patch_len, stride=stride)
        self.patch_len = patch_len
        self.stride = stride

    def forward(self, x):
        x = self.patch_embed(x)  # (B, D, N_patches)
        return x.permute(0, 2, 1)  # (B, N_patches, D)

class PatchTST(nn.Module):
    def __init__(
        self,
        time_steps: int,
        input_channels: int,
        output_channels: int,
        patch_len: int = 16,
        stride: int = 8,
        d_model: int = 128,
        num_heads: int = 4,
        num_layers: int = 3,
        dropout: float = 0.1,
        **kwargs
    ):
        super().__init__()

        self.embedding = PatchEmbedding(input_channels, patch_len, stride, d_model)
        num_patches = (time_steps - patch_len) // stride + 1
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.output_proj = nn.Sequential(
            nn.Linear(d_model, output_channels),
            nn.ReLU(),
            nn.Linear(output_channels, time_steps)  # Garantiza salida T
        )

        self.time_steps = time_steps
        self.output_channels = output_channels

    def forward(self, x):
        # x: (B, C_in, T)
        x = self.embedding(x)                      # (B, N_patches, D)
        x = x + self.pos_embedding                 # (B, N_patches, D)
        x = self.encoder(x)                        # (B, N_patches, D)
        x = x.mean(dim=1)                          # (B, D)
        x = self.output_proj(x)                    # (B, T)
        x = x.unsqueeze(1).repeat(1, self.output_channels, 1)  # (B, C_out, T)
        return x


In [20]:
import torch
import torch.nn as nn
import math

class ViT1D(nn.Module):
    """
    Vision Transformer 1D para series temporales multivariadas:
    Entrada:  (B, C_in, T)
    Salida:   (B, C_out, T)
    """
    def __init__(
        self,
        time_steps: int,
        input_channels: int,
        output_channels: int,
        patch_size: int = 8,
        d_model: int = 128,
        num_heads: int = 4,
        num_layers: int = 4,
        mlp_dim: int = 256,
        dropout: float = 0.1,
        **kwargs
    ):
        super().__init__()
        assert time_steps % patch_size == 0, "T debe ser divisible por el tamaño del patch"
        self.patch_size = patch_size
        self.num_patches = time_steps // patch_size
        self.d_model = d_model
        self.time_steps = time_steps
        self.output_channels = output_channels

        # Embedding de entrada por patch
        self.input_proj = nn.Conv1d(input_channels, d_model, kernel_size=patch_size, stride=patch_size)

        # Positional embedding
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches, d_model))
        self.dropout = nn.Dropout(dropout)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=mlp_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Proyección de salida (por patch)
        self.output_proj = nn.Linear(d_model, patch_size * output_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C_in, T)
        x = self.input_proj(x)      # (B, D, N_patches)
        x = x.permute(0, 2, 1)      # (B, N_patches, D)
        x = x + self.pos_embed      # (B, N_patches, D)
        x = self.dropout(x)
        x = self.encoder(x)         # (B, N_patches, D)
        x = self.output_proj(x)     # (B, N_patches, patch_size * C_out)

        # Reconstrucción de salida en el dominio temporal
        B, N, _ = x.shape
        x = x.view(B, N, self.output_channels, self.patch_size)         # (B, N, C_out, P)
        x = x.permute(0, 2, 1, 3).reshape(B, self.output_channels, -1)  # (B, C_out, T)

        return x  # (B, C_out, T)


In [21]:
# Configuración
B, C_in, C_out, T = 8, 163, 502, 272
x = torch.randn(B, C_in, T)

# Instancia del modelo
model = ViT1D(
    time_steps=T,
    input_channels=C_in,
    output_channels=C_out,
    patch_size=8,
    d_model=64,
    num_layers=4,
    num_heads=4,
    mlp_dim=128,
    dropout=0.1
)

# Forward pass
with torch.no_grad():
    y = model(x)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Output shape: {y.shape}")  # Esperado: (B, C_out, T)
print(f"Total trainable parameters: {total_params}")


Output shape: torch.Size([8, 502, 272])
Total trainable parameters: 480624


**UTAH DATASET**
variable	tamaño	significado habitual (Utah 2002)
ecg	18 × 141 × 1104	18 latidos × 141 electrodos × 1104 muestras
emg	18 × 221 × 1104	18 latidos × 221 g-potenciales × 1104 muestras
tag	1 × 18	índice/etiqueta de cada latido

In [ ]:
# model_registry.py

import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from types import SimpleNamespace

# ——————— Espacio de búsqueda Optuna ———————

# ──────────────────────────────────────────────────────────────────────────────
#  Hiperparámetros y Configuración (Corregido)
# ──────────────────────────────────────────────────────────────────────────────
hyperparameter_space = {
    "peepholelstm": lambda trial: {
        "hidden_size": trial.suggest_int("lstm_hidden", 16, 128, step=16),
        "num_layers": trial.suggest_int("lstm_layers", 1, 3),
        "bidirectional": trial.suggest_categorical("lstm_bidirectional", [False, True]),
    },
    "phasedlstm": lambda trial: {
        "hidden_size": trial.suggest_int("phasedlstm_hidden", 16, 128, step=16),
        "num_layers": trial.suggest_int("phasedlstm_layers", 1, 3),
        "period_init": trial.suggest_float("period_init", 0.5, 5.0, step=0.5),
        "r_on_init": trial.suggest_float("r_on_init", 0.01, 0.2, step=0.01),
    },
    "esn": lambda trial: {
        "reservoir_size":    trial.suggest_int("reservoir_size", 100, 1000, step=100),
        "spectral_radius":   trial.suggest_float("spectral_radius", 0.1, 1.5, step=0.1),
        "sparsity":          trial.suggest_float("sparsity", 0.0, 0.9, step=0.1),
        "ridge_alpha":       trial.suggest_float("ridge_alpha", 1e-6, 1e-1, log=True),
    },
    "indrnn": lambda trial: {
        "hidden_size": trial.suggest_int("indrnn_hidden", 32, 256, step=32),
        "num_layers":  trial.suggest_int("indrnn_layers", 1,   4),
        "dropout_rate":trial.suggest_float("indrnn_dropout", 0.0, 0.5, step=0.1),
        "clip_max":    trial.suggest_float("indrnn_clip",   0.5, 2.0, step=0.5),
    },
    "qrnn": lambda trial: {
        "hidden_size": trial.suggest_int("qrnn_hidden",   32, 256, step=32),
        "num_layers":  trial.suggest_int("qrnn_layers",    1,   4),
        "kernel_size": trial.suggest_int("qrnn_kernel",    2,   4),
        "dropout_rate":trial.suggest_float("qrnn_dropout",  0.0, 0.5, step=0.1),
        "pool_type":   trial.suggest_categorical("qrnn_pool", ["f", "o", "fo"]),
    },
    "bertbigru": lambda trial:{
        "hidden_dim":  trial.suggest_int("hidden_dim",   32, 256, step=32),
        "num_layers":  trial.suggest_int("qrnn_layers",    1,   4),
        "dropout_prob": trial.suggest_float("qrnn_dropout",  0.0, 0.5, step=0.1),
    },

    "transformer1d": lambda trial: {
        "d_model": trial.suggest_int("d_model", 64, 512, step=64),
        "num_heads": trial.suggest_categorical("num_heads", [2, 4, 8]),
        "num_layers": trial.suggest_int("num_layers", 2, 6),
        "dropout": trial.suggest_float("dropout", 0.1, 0.5, step=0.1),
}


}


class HyperparameterOptimizer:
    @staticmethod
    def sample_parameters(trial: optuna.Trial, variant: str) -> dict:
        if variant not in hyperparameter_space:
            raise ValueError(f"No hay espacio de hiperparámetros para '{variant}'")
        return hyperparameter_space[variant](trial)


# ——————— Fábrica de modelos ———————

class ModelFactory:
    def __init__(self, config):
        """
        config debe tener: time_steps, input_channels, output_channels
        """
        self.config = config

    def create_model(self, variant: str, params: dict) -> nn.Module:
        cls = self._get_model_class(variant)
        return cls(
            time_steps=self.config.time_steps,
            input_channels=self.config.input_channels,
            output_channels=self.config.output_channels,
            **params
        )

    def _get_model_class(self, variant: str):
        registry = {
            "peepholelstm": PeepholeLSTM,
            "phasedlstm": PhasedLSTM,
            "esn": ESN,
            "indrnn": IndRNN,
            "qrnn": QRNN,
            "bertbigru": BERT_BiGRU,
            "transformer1d": Transformer1D,
            # "tcn": TCN1D,
            # … añade aquí el resto de tus clases …
        }
        if variant not in registry:
            raise KeyError(f"Modelo desconocido '{variant}'")
        return registry[variant]


# ——————— Función objetivo para Optuna ———————

def objective(trial: optuna.Trial):
    # 1) Elige la variante
    variant = "transformer1d"

    # 2) Genera los hiperparámetros para este trial
    params = HyperparameterOptimizer.sample_parameters(trial, variant)

    # 3) Define tu configuración del problema
    config = SimpleNamespace(
        time_steps=1104,
        input_channels=141,
        output_channels=141
    )

    # 4) Crea el modelo
    factory = ModelFactory(config)
    model = factory.create_model(variant, params)

    # 5) Simula un batch de datos y calcula una loss de ejemplo
    x = torch.randn(8, config.input_channels, config.time_steps)
    y_true = torch.randn(8, config.output_channels, config.time_steps)
    y_pred = model(x)
    # imprimir tamaños
    print("Input shape:", x.shape)
    print("Output shape:", y_pred.shape)
    print("Target shape:", y_true.shape)

    loss = nn.MSELoss()(y_pred, y_true)

    return loss.item()

if __name__ == "__main__":
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=50)
    print("Mejores hiperparámetros:", study.best_params)

In [ ]:
!pip install "ray[tune]"


In [ ]:
# compare_hpo.py

import torch
import torch.nn as nn
import torch.optim as optim
import optuna
from types import SimpleNamespace

from ray import tune
from ray.tune.schedulers import ASHAScheduler
from ray.tune import CLIReporter

# ─────────── Shared setup ───────────

SEED = 42
torch.manual_seed(SEED)

B, C, T = 8, 141, 1104
x_test = torch.randn(B, C, T)
y_test = torch.randn(B, C, T)

problem = SimpleNamespace(time_steps=T, input_channels=C, output_channels=C)

# ─────────── Model definition ───────────

class MLP(nn.Module):
    def __init__(self,
                 time_steps, input_channels, output_channels,
                 stage1_out=128, stage2_out=128, stage3_out=128,
                 dropout_rates=[0.1,0.2,0.2,0.3]):
        super().__init__()
        self.time_steps = time_steps
        self.output_channels = output_channels
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Dropout(dropout_rates[0]),
            nn.Linear(input_channels*time_steps, stage1_out), nn.ReLU(),
            nn.Dropout(dropout_rates[1]),
            nn.Linear(stage1_out, stage2_out),          nn.ReLU(),
            nn.Dropout(dropout_rates[2]),
            nn.Linear(stage2_out, stage3_out),          nn.ReLU(),
            nn.Dropout(dropout_rates[3]),
            nn.Linear(stage3_out, output_channels*time_steps)
        )

    def forward(self, x):
        x = self.flatten(x)
        out = self.layers(x)
        return out.view(-1, self.output_channels, self.time_steps)

# ─────────── Optuna objective ───────────

def optuna_objective(trial):
    params = {
        "stage1_out": trial.suggest_int("stage1_out", 32, 128, step=32),
        "stage2_out": trial.suggest_int("stage2_out", 32, 128, step=32),
        "stage3_out": trial.suggest_int("stage3_out", 32, 128, step=32),
        "dropout_rates": [
            trial.suggest_float("d0", 0.0, 0.5, step=0.1),
            trial.suggest_float("d1", 0.0, 0.5, step=0.1),
            trial.suggest_float("d2", 0.0, 0.5, step=0.1),
            trial.suggest_float("d3", 0.0, 0.5, step=0.1),
        ],
    }
    # re-seed for deterministic behavior
    torch.manual_seed(SEED)
    model = MLP(problem.time_steps, problem.input_channels, problem.output_channels, **params)
    opt   = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    model.train()
    for _ in range(10):
        opt.zero_grad()
        y_pred = model(x_test)
        loss = loss_fn(y_pred, y_test)
        loss.backward()
        opt.step()
    return loss.item()

# ─────────── Ray Tune trainable ───────────

search_space = {
    "stage1_out":    tune.choice([32, 64, 96, 128]),
    "stage2_out":    tune.choice([32, 64, 96, 128]),
    "stage3_out":    tune.choice([32, 64, 96, 128]),
    "dropout_input": tune.uniform(0.0, 0.5),
    "dropout_h1":    tune.uniform(0.0, 0.5),
    "dropout_h2":    tune.uniform(0.0, 0.5),
    "dropout_out":   tune.uniform(0.0, 0.5),
}

def train_mlp_tune(config):
    drop_rates = [
        config["dropout_input"],
        config["dropout_h1"],
        config["dropout_h2"],
        config["dropout_out"],
    ]
    # re-seed
    torch.manual_seed(SEED)
    model = MLP(problem.time_steps, problem.input_channels, problem.output_channels,
                stage1_out=config["stage1_out"],
                stage2_out=config["stage2_out"],
                stage3_out=config["stage3_out"],
                dropout_rates=drop_rates)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    model.train()
    for epoch in range(10):
        opt.zero_grad()
        y_pred = model(x_test)
        loss = loss_fn(y_pred, y_test)
        loss.backward()
        opt.step()
        # **correct** way to call report in this Ray version:
        tune.report({"loss": loss.item()})

# ─────────── Main ───────────

if __name__ == "__main__":
    # 1) Optuna
    study = optuna.create_study(direction="minimize")
    study.optimize(optuna_objective, n_trials=20)
    print(">>> Optuna best params:", study.best_params)
    print(">>> Optuna best loss  :", study.best_value)

    # 2) Ray Tune
    scheduler = ASHAScheduler(metric="loss", mode="min", max_t=10, grace_period=1)
    reporter  = CLIReporter(metric_columns=["loss", "training_iteration"])
    analysis = tune.run(
        train_mlp_tune,
        config=search_space,
        num_samples=20,
        scheduler=scheduler,
        progress_reporter=reporter,
        resources_per_trial={"cpu":1},
        verbose=1
    )
    best_cfg = analysis.get_best_config(metric="loss", mode="min")
    best_trial = analysis.get_best_trial(metric="loss", mode="min")
    print(">>> Tune  best config:", best_cfg)
    print(">>> Tune  best loss  :", best_trial.last_result["loss"])

In [ ]:
best_trial = analysis.get_best_trial(metric="loss", mode="min")
print("Best loss:", best_trial.last_result["loss"])